In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap


def process_model(
    model,
    scaler,
    numerical_features,
    explainer_str,
    selected_feature_sets: list[str] | str,
    datasets_dir: str = "../../../data/predictor/cof_matched/rth_0.99",
):
    def resolve_feature_set_name(feature_set_name: str, available_feature_files: set[str]) -> str:
        stem = feature_set_name.replace(".csv", "")
        print(feature_set_name, stem, stem in available_feature_files)
        if stem not in available_feature_files:
            raise ValueError(f"Feature set '{feature_set_name}' was not found in '{datasets_dir}/features'.")
        return stem

    def build_combined_dataset(
        feature_sets: list[str],
        target_column: str = "capacity_max",
        id_column: str = "smiles",
    ) -> pd.DataFrame:
        if len(feature_sets) == 0:
            raise ValueError("selected_feature_sets cannot be empty.")

        feature_dir = os.path.join(datasets_dir, "features")
        available_files = {file_name.replace(".csv", "") for file_name in os.listdir(feature_dir)}
        resolved_names = [resolve_feature_set_name(name, available_files) for name in feature_sets]
        loaded_sets = {name: pd.read_csv(os.path.join(feature_dir, f"{name}.csv")) for name in resolved_names}

        first_df = loaded_sets[resolved_names[0]]
        keep_columns = [target_column]
        if id_column in first_df.columns:
            keep_columns.insert(0, id_column)
        combined = first_df[keep_columns].copy()

        selected_columns: list[str] = []
        for feature_set_name in resolved_names:
            dataset_part = loaded_sets[feature_set_name]
            candidate_columns = [c for c in dataset_part.columns if c not in {target_column, id_column}]
            new_columns = [c for c in candidate_columns if c not in selected_columns]

            if len(new_columns) == 0:
                continue

            if id_column in combined.columns and id_column in dataset_part.columns:
                aligned = dataset_part.set_index(id_column).reindex(combined[id_column])
                combined[new_columns] = aligned[new_columns].to_numpy()
            else:
                if len(dataset_part) != len(combined):
                    raise ValueError(f"Cannot align feature set '{feature_set_name}' without id column '{id_column}'.")
                combined[new_columns] = dataset_part.reset_index(drop=True)[new_columns]

            selected_columns.extend(new_columns)

        return combined

    if isinstance(selected_feature_sets, str):
        selected_feature_sets = [name for name in selected_feature_sets.split("__") if name]

    dataset = build_combined_dataset(selected_feature_sets)
    X = dataset.drop(columns=["capacity_max", "smiles"])
    X = X[numerical_features]

    X_columns = X.columns.tolist()
    X_values = X.values

    X_values_norm = scaler.transform(X_values)
    if explainer_str == "tree":
        explainer = shap.TreeExplainer(model)
        f_imp_all = explainer.shap_values(X_values_norm)
    elif explainer_str == "linear":
        explainer = shap.LinearExplainer(model, X_values_norm)
        f_imp_all = explainer.shap_values(X_values_norm)
    else:
        raise ValueError(f"explainer_str '{explainer_str}' is not supported.")

    explanation = shap.Explanation(values=f_imp_all, data=X_values, feature_names=X_columns)
    return explanation


from typing import Literal, Sequence

import matplotlib.cm as cm
import matplotlib.colors as mcolors


def _get_red_blue_cmap() -> mcolors.LinearSegmentedColormap:
    """Creates a red-blue colormap with a smooth transition from blue to red.

    Returns:
        A colormap object that transitions from blue to red.
    """
    gray_rgb = np.array([0.51615537, 0.51615111, 0.5161729])

    cdict: dict[Literal["red", "green", "blue", "alpha"], Sequence[tuple[float, float, float]]] = {
        "red": [
            (0.0, 0.0, 0.0),
            (0.494949494949495, 0.6035590338007161, 0.6035590338007161),
            (1.0, 1.0, 1.0),
        ],
        "green": [
            (0.0, 0.5433775692459107, 0.5433775692459107),
            (0.494949494949495, 0.14541587318267168, 0.14541587318267168),
            (1.0, 0.0, 0.0),
        ],
        "blue": [
            (0.0, 0.983379062301401, 0.983379062301401),
            (0.494949494949495, 0.6828490076357064, 0.6828490076357064),
            (1.0, 0.31796406298163893, 0.31796406298163893),
        ],
        "alpha": [(0, 1.0, 1.0), (0.494949494949495, 1.0, 1.0), (1.0, 1.0, 1.0)],
    }
    red_blue = mcolors.LinearSegmentedColormap("red_blue", cdict)
    red_blue.set_bad(gray_rgb.tolist(), 1.0)
    red_blue.set_over(gray_rgb.tolist(), 1.0)
    red_blue.set_under(gray_rgb.tolist(), 1.0)
    return red_blue


def _format_feature_name(feature_name: str) -> str:
    functional_group_mapping = [
        "aliphatic carboxylic acids",
        "aliphatic hydroxyl groups",
        "aliphatic hydroxyl groups excluding tert-OH",
        "N functional groups attached to aromatics",
        "Aromatic carboxylic acide",
        "aromatic nitrogens",
        "aromatic amines",
        "aromatic hydroxyl groups",
        "carboxylic acids COO",
        "carboxylic acids COO2",
        "carbonyl O",
        "carbonyl O, excluding COOH",
        "thiocarbonyl",
        "C(OH)CCN-Ctert-alkyl or  C(OH)CCNcyclic",
        "Imines",
        "Tertiary amines",
        "Secondary amines",
        "Primary amines",
        "hydroxylamine groups",
        "XCCNR groups",
        "tert-alicyclic amines (no heteroatoms, not quinine-like bridged N)",
        "H-pyrrole nitrogens",
        "thiol groups",
        "aldehydes",
        "alkyl carbamates (subject to hydrolysis)",
        "alkyl halides",
        "Allylic oxidation sites excluding steroid dienone",
        "amides",
        "amidine groups",
        "anilines",
        "aryl methyl sites for hydroxylation",
        "azide groups",
        "azo groups",
        "barbiturate groups",
        "Benzene rings",
        "benzodiazepines with no additional fused rings",
        "Bicyclic",
        "diazo groups",
        "dihydropyridines",
        "epoxide rings",
        "esters",
        "ether oxygens (including phenoxy)",
        "furan rings",
        "guanidine groups",
        "halogens",
        "hydrazine groups",
        "hydrazone groups",
        "imidazole rings",
        "imide groups",
        "isocyanates",
        "isothiocyanates",
        "ketones",
        "ketones excluding diaryl, a,b-unsat. dienones, heteroatom on Calpha",
        "beta lactams",
        "cyclic esters (lactones)",
        "methoxy groups -OCH3",
        "morpholine rings",
        "nitriles",
        "nitro groups",
        "nitro benzene ring substituents",
        "non-ortho nitro benzene ring substituents",
        "nitroso groups, excluding NO2",
        "oxazole rings",
        "oxime groups",
        "para-hydroxylation sites",
        "phenols",
        "phenolic OH excluding ortho intramolecular Hbond substituents",
        "phosphoric acid groups",
        "phosphoric ester groups",
        "piperdine rings",
        "piperzine rings",
        "primary amides",
        "primary sulfonamides",
        "pyridine rings",
        "quaternary nitrogens",
        "thioether",
        "sulfonamides",
        "sulfone groups",
        "terminal acetylenes",
        "tetrazole rings",
        "thiazole rings",
        "thiocyanates",
        "thiophene rings",
        "unbranched alkanes of at least 4 members (excludes halogenated alkanes)",
        "urea groups",
    ]
    from rdkit.Chem.EState.AtomTypes import _rawD

    estate_names_mapping = [name for name, smarts in _rawD]
    estate_names_mapping = {f"estate_{i}": n for i, n in enumerate(estate_names_mapping)}
    functional_group_mapping = {f"functional_groups_{i}": name for i, name in enumerate(functional_group_mapping)}
    if feature_name.startswith("estate_"):
        suffix = feature_name.removeprefix("estate_")
        return estate_names_mapping[feature_name]
        # return f'EState {suffix}'

    if feature_name.startswith("ecfp_"):
        suffix = feature_name.removeprefix("ecfp_")
        return f"ECFP {suffix}"

    if feature_name.startswith("functional_groups_"):
        suffix = feature_name.removeprefix("functional_groups_")
        if functional_group_mapping is None:
            return f"Functional group {suffix}"

        mapped_name = functional_group_mapping.get(suffix) or functional_group_mapping.get(feature_name) or functional_group_mapping.get(int(suffix)) if suffix.isdigit() else None
        return mapped_name if mapped_name else f"Functional group {suffix}"

    return feature_name.capitalize()


def _format_feature_names(
    feature_names: list[str],
) -> list[str]:
    return [_format_feature_name(name) for name in feature_names]


def shap_correlations(shap_explanation, data_columns, k=10):
    data_values = pd.DataFrame(shap_explanation.data, columns=data_columns)
    shap_values = pd.DataFrame(data=shap_explanation.values, columns=data_columns)
    absolute_means = shap_values.abs().mean(axis=0)
    absolute_means = absolute_means.reset_index(drop=False)
    absolute_means.rename(columns={"index": "feature", 0: "value"}, inplace=True)
    absolute_means = absolute_means.sort_values(by="value").tail(k)
    correlations = []
    for fname in absolute_means["feature"]:
        fvalues = data_values[fname]
        svalues = shap_values[fname]
        corr = fvalues.corr(svalues)
        correlations.append(corr)
    absolute_means["corr"] = correlations

    values = absolute_means["value"]
    feature_names = _format_feature_names(
        absolute_means["feature"].tolist(),
    )

    # 3. Plotting
    cmap = plt.get_cmap("RdBu")
    norm = mcolors.Normalize(vmin=-1.0, vmax=1.0)
    bar_colors = [cmap(norm(c)) for c in absolute_means["corr"]]

    fig, ax = plt.subplots(figsize=(9, len(values) * 0.6))
    y_pos = np.arange(len(values))

    bars = ax.barh(y_pos, values, color=bar_colors, height=0.7)

    ax.xaxis.set_ticks_position("bottom")
    ax.yaxis.set_ticks_position("none")
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(feature_names, fontsize=13)
    ax.set_xlabel("mean(|SHAP value|)", fontsize=13)

    # Add horizontal dotted lines behind bars
    for i in y_pos:
        ax.axhline(i, color="#888888", lw=0.5, dashes=(1, 5), zorder=-1)

    # 5. Add text labels at the end of bars
    max_val = np.max(values)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + (max_val * 0.01), bar.get_y() + bar.get_height() / 2, f"{val:.2f}", va="center", ha="left", fontsize=12, color="black")

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, aspect=50, ticks=[-1, -0.5, 0, 0.5, 1])
    cbar.set_label("Feature vs SHAP Correlation", fontsize=11)
    cbar.outline.set_visible(False)

    # Ensure there is room for the labels
    ax.set_xlim(0, max_val * 1.2)
    return ax


#
# ax = shap_correlations(rf_explanation, rf_explanation.feature_names, k=10)
# plt.tight_layout()

In [ ]:
import joblib
import yaml


def load_model_and_scaler(config_path: str = "brr_best_model/config.yaml"):
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    model = joblib.load(cfg["model"])
    scaler = joblib.load(cfg["scaler"])
    numerical_features = pd.read_csv(cfg["numerical_features"], header=None).iloc[:, 0].tolist()
    dataset_cfg = cfg["dataset"]
    # read features_to_keep file if present
    if "features_to_keep" in dataset_cfg:
        dataset_cfg["features_to_keep"] = pd.read_csv(dataset_cfg["features_to_keep"], header=None).iloc[:, 0].tolist()
    return model, scaler, numerical_features

In [ ]:
# rf model
rf_model, rf_scaler, rf_features = load_model_and_scaler("rf_best_model/config.yaml")
rf_datasets = ["features_rth_0.99_estate_sum", "features_rth_0.99_functional_groups_True"]
rf_expl = process_model(
    rf_model,
    rf_scaler,
    rf_features,
    "tree",
    rf_datasets,
)
ax = shap_correlations(rf_expl, rf_expl.feature_names, k=10)
plt.tight_layout()
plt.savefig("images/shap_importance_rf_whole.svg", dpi=500)

In [ ]:
# brr model
brr_model, brr_scaler, brr_features = load_model_and_scaler("brr_best_model/config.yaml")
brr_datasets = ["features_rth_0.99_ecfp_2_1024_True", "features_rth_0.99_functional_groups_True", "features_rth_0.99_descriptor_False_False", "features_rth_0.99_estate_sum"]
brr_expl = process_model(
    brr_model,
    brr_scaler,
    brr_features,
    "linear",
    brr_datasets,
)
ax = shap_correlations(brr_expl, brr_expl.feature_names, k=10)
plt.tight_layout()
plt.savefig("images/shap_importance_brr_whole.svg", dpi=500)